# X05 Knowledge Neurons


## 目标

把“知识”局部化到一组更可干预的激活单元，是从 feature 读出走向局部编辑与归因的另一条经典线。


## 为什么现在学这篇

这篇能迫使你区分“一个现象可被某些单元预测”与“这个现象真的由这些单元承载”之间的差别。


## Notebook 与交付

- 原文: https://arxiv.org/abs/2104.08696
- Notebook：`notebooks/extensions/zh/x05_knowledge_neurons.ipynb`
- 先修: X04, M04
- 在 Colab 里复现一个教学版 knowledge-neuron 打分与消融实验，并写 1 段说明高分 neuron 为什么仍然不自动等于因果解释。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import torch
from torch import nn

prompts = torch.eye(4)
probe = prompts[0:1].clone().requires_grad_(True)
model = nn.Sequential(
    nn.Linear(4, 6),
    nn.ReLU(),
    nn.Linear(6, 4),
)
with torch.no_grad():
    model[0].weight.zero_()
    model[0].bias.zero_()
    model[0].weight[0, 0] = 4.0
    model[0].weight[1, 1] = 4.0
    model[0].weight[2, 2] = 4.0
    model[0].weight[3, 3] = 4.0
    model[2].weight.zero_()
    model[2].bias.zero_()
    model[2].weight[0, 0] = 2.5
    model[2].weight[1, 1] = 2.5
    model[2].weight[2, 2] = 2.5
    model[2].weight[3, 3] = 2.5
    model[2].weight[0, 4] = 0.4
    model[2].weight[1, 5] = 0.4

hidden = model[1](model[0](probe))
logits = model[2](hidden)
target_logit = logits[0, 0]
grads = torch.autograd.grad(target_logit, hidden)[0]
scores = (hidden * grads).detach().flatten()
top_idx = int(scores.abs().argmax())

with torch.no_grad():
    original = model(prompts[0:1]).softmax(dim=-1).flatten()
    hidden_ablated = hidden.detach().clone()
    hidden_ablated[0, top_idx] = 0.0
    ablated = model[2](hidden_ablated).softmax(dim=-1).flatten()

print("Knowledge-neuron scores:", [round(float(x), 4) for x in scores])
print("Top-scoring hidden unit:", top_idx)
print("Original distribution:", [round(float(x), 4) for x in original])
print("After ablating top unit:", [round(float(x), 4) for x in ablated])


## 小结

高分 neuron 是值得继续检验的线索，不是自动成立的因果解释。


## Colab 优先的复现流程


### 运行前

- 在运行前先写 1 条你对机制的预测。
- 先写清你要对比的 baseline 是什么。
- 先决定什么结果会推翻你最喜欢的解释。


### 运行后

- 在笔记里把 observation 和 inference 分开。
- 标出复现之后仍然存在的 1 个歧义。
- 写 1 个能降低该歧义的下一步实验。


### 最后交付这些产物

- 在 Colab 里复现一个教学版 knowledge-neuron 打分与消融实验，并写 1 段说明高分 neuron 为什么仍然不自动等于因果解释。
- 1 份 experiment log，写清你改了哪些设置。
- 1 段“这次复现仍然不能证明什么”。


## 验收题

- 为什么一个 neuron 分数很高，仍然可能只是相关线索而不是承载位点？
- 在你的 toy 消融里，哪种结果最支持“局部承载”的解释？
- knowledge neurons 和 feature probes 在解释力度上最大的差别是什么？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就回去重跑复现并重写 memo。
